In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import json
import random
import logging
import re

tf.get_logger().setLevel('ERROR')

In [ ]:
#Dataset Cleaning 

df_data = pd.read_json("ner.json", lines=True)
df_data = df_data.drop(['extras'], axis=1)
df_data['content'] = df_data['content'].str.replace("\n", " ")

In [ ]:
df_data.head()

In [ ]:
df_data.iloc[0]['annotation']

In [ ]:
def mergeIntervals(intervals):
    sorted_by_lower_bound = sorted(intervals, key=lambda tup: tup[0])
    merged = []

    for higher in sorted_by_lower_bound:
        if not merged:
            merged.append(higher)
        else:
            lower = merged[-1]
            if higher[0] <= lower[1]:
                if lower[2] is higher[2]:
                    upper_bound = max(lower[1], higher[1])
                    merged[-1] = (lower[0], upper_bound, lower[2])
                else:
                    if lower[1] > higher[1]:
                        merged[-1] = lower
                    else:
                        merged[-1] = (lower[0], higher[1], higher[2])
            else:
                merged.append(higher)
    return merged

In [ ]:
def get_entities(df):
    
    entities = []
    
    for i in range(len(df)):
        entity = []
    
        for annot in df['annotation'][i]:
            try:
                ent = annot['label'][0]
                start = annot['points'][0]['start']
                end = annot['points'][0]['end'] + 1
                entity.append((start, end, ent))
            except:
                pass
    
        entity = mergeIntervals(entity)
        entities.append(entity)
    
    return entities

In [ ]:
df_data['entities'] = get_entities(df_data)
df_data.head()

In [ ]:
def convert_dataturks_to_spacy(dataturks_JSON_FilePath):
    try:
        training_data = []
        lines=[]
        with open(dataturks_JSON_FilePath, 'r') as f:
            lines = f.readlines()

        for line in lines:
            data = json.loads(line)
            text = data['content'].replace("\n", " ")
            entities = []
            data_annotations = data['annotation']
            if data_annotations is not None:
                for annotation in data_annotations:
                    #only a single point in text annotation.
                    point = annotation['points'][0]
                    labels = annotation['label']
                    # handle both list of labels or a single label.
                    if not isinstance(labels, list):
                        labels = [labels]

                    for label in labels:
                        point_start = point['start']
                        point_end = point['end']
                        point_text = point['text']
                        
                        lstrip_diff = len(point_text) - len(point_text.lstrip())
                        rstrip_diff = len(point_text) - len(point_text.rstrip())
                        if lstrip_diff != 0:
                            point_start = point_start + lstrip_diff
                        if rstrip_diff != 0:
                            point_end = point_end - rstrip_diff
                        entities.append((point_start, point_end + 1 , label))
            training_data.append((text, {"entities" : entities}))
        return training_data
    except Exception as e:
        logging.exception("Unable to process " + dataturks_JSON_FilePath + "\n" + "error = " + str(e))
        return None

def trim_entity_spans(data: list) -> list:
    """Removes leading and trailing white spaces from entity spans.

    Args:
        data (list): The data to be cleaned in spaCy JSON format.

    Returns:
        list: The cleaned data.
    """
    invalid_span_tokens = re.compile(r'\s')

    cleaned_data = []
    for text, annotations in data:
        entities = annotations['entities']
        valid_entities = []
        for start, end, label in entities:
            valid_start = start
            valid_end = end
            while valid_start < len(text) and invalid_span_tokens.match(
                    text[valid_start]):
                valid_start += 1
            while valid_end > 1 and invalid_span_tokens.match(
                    text[valid_end - 1]):
                valid_end -= 1
            valid_entities.append([valid_start, valid_end, label])
        cleaned_data.append([text, {'entities': valid_entities}])
    return cleaned_data  

In [ ]:
data = trim_entity_spans(convert_dataturks_to_spacy("ner.json"))

In [ ]:
from tqdm import tqdm
def clean_dataset(data):
    cleanedDF = pd.DataFrame(columns=["setences_cleaned"])
    sum1 = 0
    for idx in tqdm(range(len(data))):
        text = data[idx][0]
        annotations = {}
        if isinstance(data[idx][1], dict):
            annotations = data[idx][1]
        elif isinstance(data[idx][1], list) and len(data[idx][1])>0 and isinstance(data[idx][1][0], dict):
            annotations = data[idx][1][0]
        words = text.split()
        emptyList = ["Empty"] * len(words)
        numberOfWords = 0
        entities = annotations.get('entities', []) if isinstance(annotations, dict) else []
        # Normalize entity tuples/lists to (start,end,label)
        norm_entities = []
        for ent in entities:
            if len(ent) >= 3:
                norm_entities.append((int(ent[0]), int(ent[1]), ent[2]))
        # Walk through words and assign labels based on character spans
        pos = 0
        for w_i, word in enumerate(words):
            while pos < len(text) and text[pos].isspace():
                pos += 1
            word_start = pos
            word_end = word_start + len(word)
            for ent_start, ent_end, ent_label in norm_entities:
                if word_start >= ent_start and word_end <= ent_end:
                    emptyList[w_i] = ent_label
                    break
            pos = word_end
            numberOfWords += 1
        # Use .loc to append a new row instead of deprecated DataFrame.append
        cleanedDF.loc[len(cleanedDF)] = [emptyList]
        sum1 = sum1 + numberOfWords
    return cleanedDF

In [ ]:
cleanedDF = clean_dataset(data)

In [ ]:
cleanedDF.head()

### Padding and Generating Tags
Now, it is time to generate a list of unique tags that will match the named-entities.

In [ ]:
unique_tags = set(cleanedDF['setences_cleaned'].explode().unique())#pd.unique(cleanedDF['setences_cleaned'])#set(tag for doc in cleanedDF['setences_cleaned'].values.tolist() for tag in doc)
tag2id = {tag: id for id, tag in enumerate(unique_tags)}
id2tag = {id: tag for tag, id in tag2id.items()}

In [ ]:
unique_tags

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
MAX_LEN = 512
labels = cleanedDF['setences_cleaned'].values.tolist()

tags = pad_sequences([[tag2id.get(l) for l in lab] for lab in labels],
                     maxlen=MAX_LEN, value=tag2id["Empty"], padding="post",
                     dtype="long", truncating="post")

In [ ]:
tags

In [ ]:
#Tokenize
gpus = tf.config.list_physical_devices('GPU')
print(gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_virtual_device_configuration(gpu,[tf.config.experimental.VirtualDeviceConfiguration(memory_limit=4096)])

In [ ]:
from transformers import DistilBertTokenizerFast #, TFDistilBertModel
tokenizer = DistilBertTokenizerFast.from_pretrained('tokenizer/')

<a name='ex-1'></a>
### tokenize_and_align_labels

The function should perform the following:
* The tokenizer cuts sequences that exceed the maximum size allowed by your model with the parameter `truncation=True`
* Aligns the list of tags and labels with the tokenizer `word_ids` method returns a list that maps the subtokens to the original word in the sentence and special tokens to `None`. 
* Set the labels of all the special tokens (`None`) to -100 to prevent them from affecting the loss function. 
* Label of the first subtoken of a word and set the label for the following subtokens to -100. 

In [ ]:
label_all_tokens = True
def tokenize_and_align_labels(tokenizer, examples, tags):
    tokenized_inputs = tokenizer(examples, truncation=True, is_split_into_words=False, padding='max_length', max_length=512)
    labels = []
    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
#Create the train and test datasets 

test = tokenize_and_align_labels(tokenizer, df_data['content'].values.tolist(), tags)
train_dataset = tf.data.Dataset.from_tensor_slices((
    test['input_ids'],
    test['labels']
))

In [ ]:
from transformers import TFDistilBertForTokenClassification

local_model_dir = 'model/'
try:
    model = TFDistilBertForTokenClassification.from_pretrained(local_model_dir, num_labels=len(unique_tags))
except OSError:
    print(f"No local model weights found in {local_model_dir}; downloading 'distilbert-base-uncased' instead.")
    try:
        model = TFDistilBertForTokenClassification.from_pretrained('distilbert-base-uncased', num_labels=len(unique_tags))
    except TypeError:
        print('Encountered TypeError loading TF weights; retrying by converting PyTorch weights (from_pt=True).')
        model = TFDistilBertForTokenClassification.from_pretrained('distilbert-base-uncased', num_labels=len(unique_tags), from_pt=True)

In [ ]:
#Optimization
optimizer = 'adam'
model.compile(optimizer=optimizer, loss=model.hf_compute_loss, metrics=['accuracy'])
try:
    model.optimizer.learning_rate.assign(1e-5)
except Exception:
    pass
model.fit(train_dataset.batch(4),
          epochs=10, 
          batch_size=4)

In [ ]:
text = "Manisha Bharti. 3.5 years of professional IT experience in Banking and Finance domain"
inputs = tokenizer(text, return_tensors="tf", truncation=True, is_split_into_words=False, padding="max_length", max_length=512 )
input_ids = inputs["input_ids"]
#inputs["labels"] = tf.reshape(tf.constant([1] * tf.size(input_ids).numpy()), (-1, tf.size(input_ids)))

In [ ]:
output = model(inputs).logits
prediction = np.argmax(output, axis=2)
print( prediction)

In [ ]:
model(inputs)

In [ ]:
pred_labels = []

In [ ]:
%pip install seqeval

In [ ]:
true_labels = [[id2tag.get(true_index, "Empty") for true_index in test['labels'][i]] for i in range(len(test['labels']))]
np.array(true_labels).shape

In [ ]:
output = model.predict(train_dataset)

In [ ]:
predictions = np.argmax(output['logits'].reshape(220, -1, 12), axis=-1)

In [ ]:
predictions.shape

In [ ]:
from matplotlib import pyplot as plt 

p = plt.hist(np.array(true_labels).flatten())
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
from collections import Counter
Counter(np.array(true_labels).flatten())

In [ ]:
pred_labels = [[id2tag.get(index, "Empty") for index in predictions[i]] for i in range(len(predictions))]
p = plt.hist(np.array(pred_labels).flatten())
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
from seqeval.metrics import classification_report
print(classification_report(true_labels, pred_labels))